# 🧭 Chapter 07-03a: Advanced LangGraph — Multi-Agent, Streaming & Deployment

---

## 📖 The Story: The Orchestra Conductor

A symphony orchestra has a conductor who doesn't play any instrument. Instead, they coordinate: *"Violins — your turn! Now brass, come in softly. Percussion, hold..."*

A **multi-agent graph** works the same way. A supervisor node orchestrates specialist agents, streams their outputs, and handles errors gracefully.

---

## 🎯 Multi-Agent Graphs in LangGraph

```
╔══════════════════════════════════════════════════════════════════════╗
║  MULTI-AGENT GRAPH                                                  ║
║                                                                      ║
║  Multiple specialized agents in ONE graph, coordinated by a        ║
║  supervisor or through structured handoffs.                        ║
║                                                                      ║
║  Pattern: Supervisor → delegates to Agent A, B, C → collects      ║
╚══════════════════════════════════════════════════════════════════════╝
```

```
         ┌──────────────┐
         │  Supervisor  │
         └──┬───┬───┬───┘
            │   │   │
    ┌───────▼┐ ┌▼──┐ ┌▼───────┐
    │Researcher│ │Writer│ │Reviewer│
    └───────┬┘ └┬──┘ └┬───────┘
            │   │   │
         ┌──▼───▼───▼───┐
         │  Supervisor  │
         └──────┬───────┘
                │
           ┌────▼────┐
           │   END   │
           └─────────┘
```

---

## 🌊 Streaming in LangGraph

Three streaming modes:

### 1. Stream Events (most common)
```python
for event in app.stream({"messages": [msg]}, config, stream_mode="values"):
    # event = full state after each node
    print(event["messages"][-1])
```

### 2. Stream Updates (per-node changes)
```python
for node_name, update in app.stream({"messages": [msg]}, config, stream_mode="updates"):
    print(f"Node '{node_name}' produced: {update}")
```

### 3. Token-by-Token (astream_events)
```python
async for event in app.astream_events({"messages": [msg]}, config, version="v2"):
    if event["event"] == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="")
```

---

## ⚡ Error Handling Patterns

```python
# Pattern 1: Try/except in node
def safe_tool_node(state):
    try:
        return execute_tools(state)
    except Exception as e:
        return {"messages": [AIMessage(content=f"Tool error: {e}")]}

# Pattern 2: Retry with conditional edge
def should_retry(state) -> Literal["retry", "end"]:
    if state.get("error_count", 0) < 3:
        return "retry"
    return "end"

# Pattern 3: Fallback node
graph.add_node("fallback", lambda s: {"messages": [AIMessage(content="I encountered an error. Let me try a different approach.")]})
```

---

## 🚀 Deployment Options

| Option | Best For | How |
|--------|----------|-----|
| **LangGraph Platform** | Production, managed | `langgraph deploy` |
| **FastAPI** | Self-hosted API | Wrap `app.invoke()` in endpoint |
| **Streamlit** | Demos, prototypes | Streamlit + LangGraph |
| **LangServe** | Quick API serving | `add_routes(app, chain)` |

```python
# FastAPI example
from fastapi import FastAPI
api = FastAPI()

@api.post("/chat")
async def chat(message: str, thread_id: str):
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"messages": [HumanMessage(message)]}, config)
    return {"response": result["messages"][-1].content}
```

---

## 🎓 Interview Corner

**Q: How would you design a production LangGraph agent with error recovery?**

<details>
<summary>💡 Click for Expert Answer</summary>

1. **Structure**: Supervisor → specialized agents (research, execute, validate)
2. **Error handling**: Each node wraps in try/except, returns error to state
3. **Retry logic**: Conditional edge checking error_count < max_retries
4. **Fallback**: If all retries fail, route to fallback node that explains the issue
5. **Persistence**: PostgresSaver for checkpointing, Redis for caching
6. **Monitoring**: LangSmith for tracing every node execution
7. **Streaming**: `stream_mode="values"` for real-time UI updates
8. **HITL**: `interrupt_before` for high-risk actions (sending emails, DB writes)

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 Multi-agent: Supervisor delegates to specialist agents.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 stream_mode="values"|"updates" for real-time output.           │
├─────────────────────────────────────────────────────────────────────┤
│  📌 astream_events for token-by-token streaming.                   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Error patterns: try/except, retry edges, fallback nodes.       │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Deploy via LangGraph Platform, FastAPI, or Streamlit.          │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Project — Build an Autonomous Research Agent!

---

### 🎉 Advanced LangGraph Unlocked!
Multi-agent, streaming, deployment — production-ready knowledge. 🚀